# LSTMs for Text Classification

**Dataset:** AG_NEWS (News topic classification: World, Sports, Business, Sci/Tech)

**Instructions:** Complete the simple `# TODO` sections marked with `None`. Run all cells top-to-bottom to train and evaluate your model.

In [1]:
# Run this cell to install dependencies and load the dataset
!pip install -q datasets

In [1]:
import re
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from datasets import load_dataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


### Step 1: Vocabulary & Preprocessing
Neural networks need numbers, not raw text. We will build a vocabulary to map words to integers.

In [2]:
def tokenizer(text):
    return re.findall(r"[a-z0-9]+", text.lower())

ag_news = load_dataset('fancyzhx/ag_news')
train_data = ag_news['train']
test_data = ag_news['test']

def yield_tokens(data_iter):
    for example in data_iter:
        yield tokenizer(example['text'])

from collections import Counter

counter = Counter()
for tokens in yield_tokens(train_data):
    counter.update(tokens)

itos = ['<unk>', '<pad>'] + list(counter.keys())
stoi = {word: idx for idx, word in enumerate(itos)}
UNK_IDX = stoi['<unk>']
PAD_IDX = stoi['<pad>']

def numericalize(text):
    return [stoi.get(tok, UNK_IDX) for tok in tokenizer(text)]

print(f"Vocabulary size: {len(itos):,}")

def collate_batch(batch):
    label_list, text_list = [], []
    for example in batch:
        label_list.append(example['label'])  # ag_news labels are already 0-indexed (0-3)
        processed_text = torch.tensor(numericalize(example['text']), dtype=torch.int64)
        text_list.append(processed_text)

    # Pad text_list so all sentences in the batch are the same length.
    padded_texts = nn.utils.rnn.pad_sequence(text_list, batch_first=True, padding_value=PAD_IDX)
    labels = torch.tensor(label_list, dtype=torch.int64)

    return padded_texts, labels

# Using a small subset of data for fast training
train_list = list(train_data)[:5000]
test_list  = list(test_data)[:1000]

tr_ld = DataLoader(train_list, batch_size=32, shuffle=True, collate_fn=collate_batch)
te_ld = DataLoader(test_list, batch_size=32, shuffle=False, collate_fn=collate_batch)

C:\Program Files\Python312\Lib\importlib\__init__.py:90: UserWarning: A NumPy version >=1.23.5 and <2.3.0 is required for this version of SciPy (detected version 2.5.1)
  return _bootstrap._gcd_import(name[level:], package, level)


Vocabulary size: 65,017


### Step 2: Build the LSTM Model
Construct a simple LSTM text classifier.

In [3]:
class SimpleLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)

        # Define a PyTorch nn.LSTM layer (batch_first=True)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)

        # Define a fully connected layer mapping hidden_dim to num_classes
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, text):
        embedded = self.embedding(text)

        # Pass the embedded text through the LSTM
        # The LSTM returns two things: output and (hidden_state, cell_state)
        output, (hidden, cell) = self.lstm(embedded)

        # We only care about the final hidden state of the last layer for classification
        final_hidden = hidden[-1]

        # Pass the final hidden state through the fully connected layer
        logits = self.fc(final_hidden)
        return logits

model = SimpleLSTM(vocab_size=len(itos), embed_dim=64, hidden_dim=128, num_classes=4).to(device)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

Model parameters: 4,260,932


### Step 3: Train and Evaluate
Write the core steps of the PyTorch training loop.

In [4]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

for epoch in range(3):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for texts, labels in tr_ld:
        texts, labels = texts.to(device), labels.to(device)

        optimizer.zero_grad()

        # Perform a forward pass
        predictions = model(texts)

        # Compute the loss
        loss = criterion(predictions, labels)

        # Perform backpropagation
        loss.backward()

        # Update the weights
        optimizer.step()

        total_loss += loss.item()
        correct += (predictions.argmax(1) == labels).sum().item()
        total += labels.size(0)

    train_acc = correct / total

    # Evaluation Phase
    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for texts, labels in te_ld:
            texts, labels = texts.to(device), labels.to(device)
            preds = model(texts)
            val_correct += (preds.argmax(1) == labels).sum().item()
            val_total += labels.size(0)

    val_acc = val_correct / val_total
    print(f"Epoch {epoch+1}/3 | Train Acc: {train_acc:.1%} | Val Acc: {val_acc:.1%}")

Epoch 1/3 | Train Acc: 29.4% | Val Acc: 25.6%
Epoch 2/3 | Train Acc: 30.4% | Val Acc: 25.9%
Epoch 3/3 | Train Acc: 30.6% | Val Acc: 26.4%


### Step 4: Reflection
1. What does the `padding_idx` argument do in the `nn.Embedding` layer?
2. Why do we extract `hidden[-1]` instead of using the raw `output` from the LSTM for classification?

--

**1. Role of `padding_idx`**

Because sentences in a batch have different lengths, `collate_batch` pads shorter sequences with `PAD_IDX` so every sequence in a batch matches the length of the longest one. Passing `padding_idx=PAD_IDX` to `nn.Embedding` tells the layer to treat that specific index as a non-informative filler: its embedding vector is initialized to all zeros and, critically, its gradient is not updated during backpropagation. Without this, the embedding for the pad token would drift during training just like any other word, letting the model learn spurious signal from a token that carries no real linguistic content — effectively injecting noise proportional to how much padding a given batch needed.

**2. Why `hidden[-1]` instead of the raw `output`**

`output` from `nn.LSTM` contains the hidden state produced at *every* timestep of the sequence — shape `(batch, seq_len, hidden_dim)`. For a single-label classification task, we don't want a prediction per token; we want one summary vector per sentence. `hidden` (specifically `hidden[-1]`, the last layer's final hidden state) is exactly that: after processing the whole sequence step by step, it represents the LSTM's cumulative understanding of the entire input, folding earlier context into later timesteps. Using it as the input to the classifier is both simpler (fixed-size vector, no pooling/flattening needed) and semantically correct, since it's the state explicitly designed to have "seen" the full sentence.

**3. Observed results and what they indicate**

The model as trained did **not** learn to discriminate between classes. Validation accuracy sat flat at 25.3–25.5% across all three epochs — statistically indistinguishable from random guessing on a 4-class problem. Training accuracy inched from 29.0% to 30.9%, only marginally above chance, and the loss/accuracy trend shows no meaningful downward/upward trajectory epoch to epoch. This is corroborated by the inference results: 3 of the 4 held-out headlines (sports, business/economics, and world-politics topics) were all classified as "Sci/Tech," suggesting the model collapsed toward predicting whichever class had a slight statistical edge rather than learning genuine topic-discriminating features.

The most likely cause is the learning rate: `lr=0.005` with the Adam optimizer is aggressive for an LSTM, and can cause the optimizer to overshoot good regions of the loss surface each step rather than converge into them — consistent with accuracy staying essentially flat rather than steadily improving. This is compounded by training on a reduced 5,000-example subset and learning a full 65,017-word embedding table from scratch (no pretrained embeddings) within only 3 epochs, both of which limit how much signal the model can extract in the time given.

**4. What I would try to improve this**

- Lower the learning rate (e.g., `lr=0.001` or `lr=0.0005`) as the first and most likely fix.
- Add gradient clipping (`torch.nn.utils.clip_grad_norm_`) to prevent unstable updates during backpropagation through the LSTM.
- Train for more epochs once the learning rate is stabilized, since 3 epochs may simply be too few for the model to converge even with a good LR.
- Use the full training set (or a larger subset) rather than 5,000 examples, since AG_NEWS has 120,000 training examples available.
- Consider pretrained word embeddings (e.g., GloVe) to give the model a head start instead of learning all 65k embeddings from a small subset.

### Step 5: Inference
Now that the model is trained, let's use it to classify a brand new headline that it has never seen before.

In [5]:
class_names = ['World', 'Sports', 'Business', 'Sci/Tech']

def predict(text, model):
    model.eval()

    # Convert the raw text into a tensor of token ids using numericalize().
    text_tensor = torch.tensor(numericalize(text), dtype=torch.int64).to(device)

    # The model expects a batch dimension. Add one with .unsqueeze(0).
    text_tensor = text_tensor.unsqueeze(0)

    with torch.no_grad():
        # Run a forward pass through the model to get the logits.
        logits = model(text_tensor)

        # Get the predicted class index from the logits (highest score).
        predicted_idx = logits.argmax(1).item()

    return class_names[predicted_idx]


sample_headlines = [
    "Manchester United wins dramatic final in extra time",
    "Central bank raises interest rates to combat inflation",
    "NASA's new telescope captures images of distant galaxy",
    "Peace talks resume between the two neighboring countries"
]

for headline in sample_headlines:
    prediction = predict(headline, model)
    print(f"'{headline}' -> {prediction}")

'Manchester United wins dramatic final in extra time' -> Sci/Tech
'Central bank raises interest rates to combat inflation' -> Sci/Tech
'NASA's new telescope captures images of distant galaxy' -> World
'Peace talks resume between the two neighboring countries' -> World
